## Parenthesis
for parenthese problems, usually a stack will solve the problems, but using counter to keep track of the count of ( and ) is feasbile too.

In [2]:
from typing import Optional, List

<b>20.Valid Parentheses </b>. 

using stack. loop the array, if it is the start sign, put it in stack. if it is an end side, pop the stack to get the last item, compare with the correpsonding item's start sign. If not match, return False direclty, else contingue.  
at the end, if the whole stack is empty, then True

In [3]:
def isValid( s):
    """complexity O(N), search in keys is O(1), pop for list is O(1) too"""
    mapping = {')': '(', '}': '{', ']': '['}
    n = len(s)
    if n%2==1:
        return False
    stack = []
    for item in s:
        # start of the parentheses
        if item not in mapping.keys():
            stack.append(item)
        else:
            if (not stack) or (stack.pop()!=mapping[item]):
                return False
    return len(stack)==0

In [2]:
s="()[]{}"
isValid( s)

True

In [3]:
s="(("
isValid( s)

False

<b> 22 Generate Parenthese </b>

return all valid parenthese combination give  k number of pairs.

recursive functions, it is actually like tree traversal, left (open parenthesi) first, then right (closing parenthesis) second

In [4]:

def generateParenthesis(n: int) -> List[str]:
    def helper(result, temp, n, open_ct, end_ct):
        if len(temp)==2*n:
            result.append(temp[:])
            return
        if open_ct < n:
            # able to add ( to the stack
            helper(result, temp+'(', n, open_ct+1, end_ct)
        if end_ct < open_ct:
            # able to add ) to the stack
            helper(result, temp+')', n, open_ct, end_ct+1)
    
    result = []
    helper(result, "", n, 0, 0)
    return result

<b> 32 Longest valid Parenthesis </b>

having a stack saving the invalid index that breaks a substring that forms the valid parenthsis, then using the invalid index to locate the length of those valid formation and compare their size.

time complexity:
- O(N) of looping over the string
- O(N) at worse case to check those invalid index
- so total O(N)

<b> 1249 minimum remove to make valid parentheses</b>

stack to store the items that can be removed.

time complexity
- O(N) to find the to_remove stack
- worse case O(N) to get the index needed to be removed
- O(N) to go over origianl string to get what should be kept
- O(N) to construct the res str
- in total O(N)

In [3]:
def minRemoveToMakeValid(s: str) -> str:
    to_remove = [] # stack of (parenthese, ind)
    for ind, c in enumerate(s):
        if c == "(":
            to_remove.append((c, ind))
        if c == ")":
            if to_remove and to_remove[-1][0] == "(":
                to_remove.pop()
            else:
                to_remove.append((c, ind))
    index_to_remove = {ind for _, ind in to_remove }
    res = []
    for ind, c in enumerate(s):
        if ind not in index_to_remove:
            res.append(c)
    return "".join(res)




In [4]:
minRemoveToMakeValid("lee(t(c)o)de)")

'lee(t(c)o)de'

<b> 301 remove the invalid parenthese </b>

hard part is the output all possible ways.  
here using the BFS way for easy understanding. Each time try to remove a side of the parenthsis, then check if it is a valid string. Stop generating more substring for next level if found one value in current level, that guarantees minimual removal



In [5]:
class Solution:
    def removeInvalidParentheses(self, s: str) -> List[str]:
        """
        using BFS, starting queue is the original string
        then second level queue is by removing one item. BFS can ensure the result is removing the minimum number of items.
        time complexity:
        -   is_valid() takes O(N) on average, N is string l
        -   BFS possibly explores N level, for all possible substrings. For each char of the string, either it is kept or removed, so in total there are 2**N different sub strings.
        -   each substring needs a O(N) check so in total the time complexity is O(N*2**N) for worse case.

        space complexity:
        -   there are at most 2**N number of substring, and each substring at most have N in size, so in total worse case is O(N*2**N)
        """
        def is_valid(string):
            count = 0
            for c in string:
                if c == "(": count += 1
                elif c == ")": count -= 1
                if count < 0: return False
            return count == 0
        queue = deque([s])
        visited =set([s])
        res = []
        found = False
        
        while queue:
            next_q = deque()
            for cur_s in queue: # same level
                if is_valid(cur_s):
                    res.append(cur_s)
                    found = True
                if found: 
                    # dont generate next level of queue
                    # minimum removal step achieved
                    continue
                for i in range(len(cur_s)):
                    if cur_s[i] not in ("(", ")"):
                        continue
                    if i > 0 and cur_s[i]==cur_s[i-1]:
                        # avoid deleting same char to generate duplicated next level string
                        continue
                    next_s = cur_s[:i]+cur_s[i+1:]
                    if next_s not in visited:
                        visited.add(next_s)
                        next_q.append(next_s)
            if not found:
                queue = next_q
            else:
                break
        return res



<b> 678 valid parenthesis string </b>

special requirement is: * can be used as ( or ) or empty string

In [2]:
def checkValidString(s: str) -> bool:
    """
    stack with rules,
    order matters
    """
    # range of allowed ) given ( number
    # c_min the min num of unpaired ( that must be paird
    # c_max the max num of unpaird ( when considering * as (
    # valid whhen at the end c_min = 0, cmax >=0
    c_max = 0 
    c_min = 0 
    for c in s:
        if c == "(":
            c_min += 1
            c_max += 1
        elif c == ")":
            # if c_min is already 0, then there is no unmatched ( left, it is okay to leave c_min as 0 since it is the lower bound
            c_min = max(c_min-1, 0)
            c_max -= 1
        else:
            # if * as ), it will reduce the unpaird number
            # but should not be negative
            c_min = max(c_min-1, 0)
            # if * as (, it increase the unpaird number
            c_max += 1
        # even using *, there are not enough ( to pair with )
        # so it is invalid
        if c_max < 0:
            return False
    return c_min == 0


In [3]:
checkValidString("))*")

False

<b> 2116 check if a parenthesis string can be valid </b>

the locked mask provide flexibility, so check the string from two directions.

it is the extension of using a single integer value to keep track of the ( and ) encountered, if it is negtive at some point then it is invalid string, and at the end if the value is positive it is invalid too, because it means there are some opening ( dont find an ending ).

here introducing the concept of flexible positions, so it can compromise some invalid position from the fixed position. that is why check if fixed + flexible still negative or not for each position.

time complexity: O(N) for 2 loop  
space complexity: O(N) for the duplicate of the string and lock


In [4]:

def canBeValid(s: str, locked: str) -> bool:
    n = len(s)
    if n%2 != 0:
        return False
    def helper(s, locked, sign):
        """
        fixed count checks the pair of valid (), if it is negative, then just looking for those fixed position, the string is not valid
        flexible count are positions that allow changing
        even though fixed positions are not valid, with flexible positions, it could happen that the final string can be valid after modifications.

        but need to check backwards for edge cases like "))((" "0011"
        """
        fixed = 0
        flexible = 0
        for i in range(len(s)):
            if locked[i] == "1":
                fixed += 1 if s[i] == sign else -1
            else:
                flexible += 1
            # if at any position the string is not valid, we can finish the checking
            if fixed + flexible < 0:
                return False

        return fixed <= flexible
    # test case "))((" "0011" wont pass if not check backwards
    return helper(s, locked, "(") and helper(s[::-1], locked[::-1], ")")


In [5]:
canBeValid("))((", "0011")

False

<b> 2337 move pieces to obtain a string </b>

using two points, i used to check the non _ char in start, j used to check the non _ char in target.

try to find the 100% sure negative case and break the loop. otherwise at the end return True

absolute negative case:
-   only one of i or j reaches the end, that means there are some non _ element left with nothing to pair with
-   the position of L in start is smaller than position of L in target, so start cannot move to the right.
-   the position of R in start is larger than position of R in target, so start cannot move to the left


In [6]:

def canChange(start: str, target: str) -> bool:
    n = len(start)
    i = j = 0
    # should be i<=n and j<=n instead of i<n and j<n otherwise the test case "_L" "LR" wont pass
    while i <= n and j <= n:
        while i<n and start[i] == "_": i+=1
        while j<n and target[j] == "_": j+=1
        if i == n or j == n:
            # both should all reach the end otherwise there is one string that still have non empty char left
            return i==n and j==n
        # the non empty char should match in order
        if start[i]!=target[j]: return False
        if start[i]=="L":
            # target[j] is also L, but its left should have more empty space if j >i, however, start string does not have that much empty space that can achieve it
            if j > i: return False
        else:
            # start[i] and target[j] all R, but start[i] can only move right, it cannot move to a previous location to satisfy the criteria
            if j < i: return False
        i+=1
        j+=1
    return True